# Iridium-1 — high-RAM Colab training

This notebook is deliberately usable on a **CPU-only 48 GiB system-RAM Colab**.
A GPU is optional for the default `micro` rung, which is about 1.0 B parameters.
It needs roughly 16 GB of full fp32 Adam state before activations and checkpoints.
A 48 GiB runtime has host RAM for that; it will be slow on CPU, but it does not
require GPU HBM or bf16. **It cannot full-train the 8B rung:** that has 120 GiB
of optimizer state before activations and needs a distributed training system.

**Runtime → Change runtime type → High-RAM** (GPU optional), then **Runtime → Run all**.

| rung | params | practical memory | default use |
|---|---:|---:|---|
| `nano` | 34 M | ~0.5 GB | quick smoke test |
| `nano100m` | 104 M | ~1.7 GB | short CPU run |
| `micro` / `test1b` | 1.0 B | ~16 GB + activations | 48 GiB high-RAM CPU runtime |
| `8b` | 8.07 B | ~120 GiB + activations | costed only; distributed training required |

> **CPU is intentional.** The notebook selects CUDA only when it is present;
> otherwise it trains in fp32 on CPU. The small batch below leaves headroom for
> activations and two on-disk checkpoints. Set `STEPS` higher if you are happy
> to let the CPU run for a long time.


In [ ]:
import os
from pathlib import Path

try:
    import psutil
    ram_gib = psutil.virtual_memory().total / 2**30
except ImportError:
    ram_gib = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30

print(f'system RAM: {ram_gib:.1f} GiB')
if ram_gib < 40:
    print('WARNING: select Runtime > Change runtime type > High-RAM before using micro.')

!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv || echo 'No GPU detected — CPU fp32 mode is supported.'


## 1 · Get the code


In [ ]:
!git clone --depth 1 https://github.com/sporadicstudiosind-cloud/test.git iridium 2>/dev/null || (cd iridium && git pull)
%cd iridium
!pip -q install pyyaml psutil
import sys; sys.path.insert(0, '.')
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 2 · Pick the rung

The default is the 1 B `micro` rung for a high-RAM CPU session. It does **not**
need GPU memory. Full fp32 Adam is approximately 16 GB before activations, so
48 GiB system RAM leaves working headroom; `BATCH = 1` makes that headroom
explicit. It will be much slower than a GPU run. Use `nano100m` for a quicker
end-to-end check.


In [ ]:
!python -m iridium ladder


In [ ]:
# This is the CPU/high-RAM preset. Change to 'nano100m' for a faster dry run.
# Do not choose '8b' here: a full 8B Adam run needs >120 GiB before activations.
RUNG   = 'micro'       # 'nano' 34M | 'nano100m' 104M | 'micro'/'test1b' 1.0B | '8b' 8.07B
STEPS  = 25            # safe first checkpoint; raise after confirming memory headroom
BATCH  = 1             # keep a 48 GiB CPU session comfortably below its RAM limit
LR     = 2e-4
# Full omnimodal mixture. Narrow it only if you want a faster CPU run.
MIXTURE = ('channel_depth=0.30,channel_intervention=0.25,false_premise=0.15,'
           'field_rollout=0.15,scene_goal=0.15')

from iridium.config import get_config
cfg = get_config(RUNG)
required_state_gib = cfg.training_state_bytes() / 2**30
print(cfg.report().render())
print(f'full BF16 Adam state (excluding activations): {required_state_gib:.1f} GiB')
if RUNG == '8b':
    raise RuntimeError(
        '8B full-parameter training is not a Colab/48 GiB job. This repository '
        'does not implement the required distributed dispatcher. See docs/training-8b.md.'
    )


## 3 · Check the architecture before spending CPU time

The parameter formulae that cost a 9-trillion-parameter configuration are the same
ones that describe this model. At the rungs that instantiate, the difference between
formula and modules is exactly zero — and the cache-parity gate proves that cached
decoding computes what teacher forcing trained. If either fails, stop.


In [ ]:
!python -m pytest tests/unit/test_config_inventory.py tests/integration/test_kv_parity.py -q


## 4 · Train


In [ ]:
import torch, time, json
from pathlib import Path
from iridium.model.iridium1 import Iridium1
from iridium.training.datasets import build_corpus, describe
from iridium.training.trainer import TrainConfig, Trainer
from iridium.training.losses import LossWeights

# CUDA is an optional acceleration. CPU fp32 is the supported high-system-RAM path.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
major = torch.cuda.get_device_capability()[0] if device == 'cuda' else 0
use_bf16 = device == 'cuda' and major >= 8
print(f'device={device} compute_capability={major}.x bf16={use_bf16}')
if device == 'cpu':
    print('CPU fp32 mode: system RAM, not GPU HBM, is the limiting memory resource.')

mixture = {k: float(v) for k, v in (p.split('=') for p in MIXTURE.split(','))}
train = build_corpus(40000, seed=0, split='train', mixture=mixture)
test  = build_corpus(800, seed=1000, split='test', mixture=mixture)
extra = build_corpus(400, seed=2000, split='extrapolation', mixture=mixture)
print(describe(train))

model = Iridium1(cfg).to(device)
print(f'{sum(p.numel() for p in model.parameters()):,} parameters on {device}')


In [ ]:
from iridium.evaluation.harness import evaluate

before = evaluate(model, test, max_per_family=16)
print('untrained:', json.dumps(before, indent=1)[:900])


In [ ]:
tcfg = TrainConfig(steps=STEPS, batch_size=BATCH, lr=LR, seed=0,
                   label=f'colab-{RUNG}', log_every=max(STEPS//50, 1),
                   checkpoint_every=max(STEPS//5, 1))
trainer = Trainer(model, train, tcfg, LossWeights(), out_dir=Path('runs/colab'),
                  device=device)
t0 = time.time()
history = trainer.train()
print(f'trained in {(time.time()-t0)/60:.1f} min')


## 5 · Grade it

Not loss — **graded accuracy**, from free-running generation, checked against an
independent computation: the analytic Manning law, the spectral solver, or the scene
environment's own goal predicate. Every score is reported beside the baseline a model
gets by ignoring its input entirely, because a number without a baseline cannot be read.

`extrapolation` draws discharges from a band the training split never contains, so a
score there cannot be earned by having seen a neighbouring example.


In [ ]:
model.eval()
results = {
    'interpolation': evaluate(model, test, max_per_family=48),
    'extrapolation': evaluate(model, extra, max_per_family=48),
    'untrained_baseline': before,
}
print(json.dumps(results, indent=2))

print()
print(f"{'family':<26}{'trained':>9}{'baseline':>10}{'verdict':>14}")
for fam, r in results['interpolation'].items():
    acc, base = r.get('accuracy', 0), r.get('baseline', 0)
    verdict = 'LEARNED' if acc > base + 0.1 else ('at baseline' if acc >= base else 'BELOW baseline')
    print(f'{fam:<26}{acc:>9.3f}{base:>10.3f}{verdict:>14}')


## 6 · Look at the routing

Balance and specialisation pull in opposite directions, and a router can look healthy
on either while failing the other. `I(family; stack)` separates them: zero means routing
is independent of the task, `log(min(families, stacks))` means a clean partition.


In [ ]:
from iridium.evaluation.routing import analyse
from iridium.training.datasets import BatchLoader
report = analyse(model, BatchLoader(test, cfg.codecs, 16, 0, device=device))
print(report.render())


## 7 · Save the weights

Stored fp16 so the file stays small. Mount Drive to keep it past the session.
On CPU, the conversion happens only when saving; training remains fp32.


In [ ]:
path = trainer.save('final', extra={'evaluation': results})
print('saved', path)

import torch
blob = torch.load(path, map_location='cpu', weights_only=False)
half = {k: (v.half() if v.is_floating_point() else v) for k, v in blob['state_dict'].items()}
torch.save({'state_dict': half, 'manifest': blob['manifest']}, 'iridium-fp16.pt')
print('fp16 checkpoint written')

# from google.colab import drive; drive.mount('/content/drive')
# !cp iridium-fp16.pt /content/drive/MyDrive/

from google.colab import files
files.download('iridium-fp16.pt')


## 8 · Talk to it

Ask for a number and it answers with one — beside the analytic value, so you can check
it rather than believe it. Numbers travel as typed quantities, never as decimal prose:
the same mapping learned from digit-bytes reaches 18.6% of answers inside a 2% tolerance,
and learned from typed values, 100%.


In [ ]:
import subprocess, os, threading
os.environ['IRIDIUM_CHECKPOINT'] = str(path)
os.environ['PYTHONPATH'] = '.'
os.environ['PORT'] = '8080'
threading.Thread(target=lambda: subprocess.run(['python','serve/server.py']), daemon=True).start()
import time; time.sleep(25)

import json, urllib.request
def ask(q, loops=1):
    body = json.dumps({'prompt': q, 'loops': loops}).encode()
    req = urllib.request.Request('http://127.0.0.1:8080/api/ask', body,
                                 {'Content-Type': 'application/json'})
    return json.load(urllib.request.urlopen(req, timeout=120))

for q in ['normal depth | S=0.0020 n=0.030 q=3.0',
          'depth ratio | S=0.0020 n=0.030 q=3.0 x2.0',
          'doubling the discharge doubles the flow depth']:
    r = ask(q)
    a = r['answer']
    if 'predicted' in a:
        print(f"{q}\n   model {a['predicted']:.4f}  analytic {a['analytic']:.4f}"
              f"  rel err {a['relative_error']:.4f}  within 2%: {a['within_2pct']}")
    else:
        print(f"{q}\n   verdict {a['verdict']}")
    t = r['telemetry']
    print(f"   routed to {t['stacks_used']}/{len(t['routing'])} stacks,"
          f" focus {t['mean_focus']:.3f}, {t['expected_loops']:.2f} ponder loops")


In [ ]:
from google.colab.output import eval_js
print('Probe UI:', eval_js('google.colab.kernel.proxyPort(8080)'))


---

### What a good run looks like

- `channel_depth` and `channel_intervention` **above 0.9** on interpolation. These are
  affine in log space once the inputs arrive as typed quantities, so anything much
  below that means something is wrong, not that the task is hard.
- `extrapolation` will be **lower** — that band is outside the training discharges, and
  the gap between the two columns is the honest measure of what was learned versus fitted.
- `false_premise` draws from only ten distinct claims, so a high score there is
  **memorisation, not judgement**. Do not read it as calibration.
- `field_rollout` must beat the persistence baseline (emit the input frame unchanged)
  to mean anything at all.
- Stack-usage entropy near `log(n_stacks)` means no collapse; `I(family; stack)` near
  zero means no specialisation yet — that is what phase 2 is for.
